# Local LoRA fine-tune cho baseline v8
Train ba adapter riêng từ cùng Qwen3-4B-Instruct-2507 gốc: planner chọn danh sách tool/thứ tự; caller chọn arg/value; summarizer chọn next. **Không có global fine-tune.**

Mặc định dùng base 4-bit + LoRA (QLoRA) cho T4, train tuần tự từng role để giải phóng VRAM giữa các job. Mẫu âm nằm ở rejected và các tình huống counterfactual được ToolEnv kiểm tra; không SFT vào hành động sai. Summarizer chỉ tính loss ở next, mask answer vì dataset không có nhãn câu kết luận.

Attach dữ liệu và **toàn bộ baseline_v8_finetune/**. Không cần v5/v8 làm dependency. Bộ 100 ID seed 42 từ report_v8 mặc định nằm ngoài train/validation. Không có upload hoặc push model tự động.


In [ ]:
from pathlib import Path
import sys
import json
import subprocess
import os

SCRIPT_DIR = None  # Có thể đặt thủ công tới baseline_v8_finetune
if SCRIPT_DIR is None:
    candidates = []
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates += [p.parents[3] for p in kaggle_input.rglob('src/lifemate_v8_finetune/training/train.py')]
    for ancestor in (Path.cwd(), *Path.cwd().parents):
        for p in (ancestor, ancestor/'baseline_v8_finetune'):
            if (p/'src/lifemate_v8_finetune/training/train.py').is_file():
                candidates.append(p)
    if not candidates:
        raise FileNotFoundError('Attach toàn bộ baseline_v8_finetune hoặc đặt SCRIPT_DIR.')
    SCRIPT_DIR = candidates[0]
SCRIPT_DIR = Path(SCRIPT_DIR)
for name in ('prepare_data.py','train_role.py','requirements/finetune.txt',
             'src/lifemate_v8_finetune/prompts/summarizer.txt'):
    if not (SCRIPT_DIR/name).is_file():
        raise FileNotFoundError(SCRIPT_DIR/name)

MODEL_ID = 'Qwen/Qwen3-4B-Instruct-2507'
DATA_ROOT = None  # Folder chứa train/train.jsonl; None = tự tìm trên Kaggle
WORK_DIR = Path('/kaggle/working/baseline_v8_finetune')
DATA_DIR = WORK_DIR/'data'
TRAIN_OUTPUT = WORK_DIR/'local_lora'
ADAPTER_ROOT = TRAIN_OUTPUT/'adapters'
HOLDOUT_IDS = SCRIPT_DIR/'configs/holdout_ids_seed42.json'  # None nếu tự thiết kế tập eval khác
VALIDATION_FRACTION = 0.10
SEED = 42
DATA_LIMIT = 0  # 0 = toàn bộ; có thể dùng 20 để smoke-test tạo data
NEGATIVES_PER_EXAMPLE = 2
COUNTERFACTUALS_PER_CASE = 2

CUDA_DEBUG = False  # True: child jobs use CUDA_LAUNCH_BLOCKING=1; slower, diagnostic only
TRAIN_ROLES = ['planner','caller','summarizer']
USE_4BIT = True  # False: LoRA trên base FP16/BF16, cần nhiều VRAM hơn
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
EPOCHS = 1
MAX_STEPS = -1  # -1 = theo EPOCHS; dùng 5-10 để smoke-test GPU trước
MAX_SEQ_LENGTH = 8192
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 16
LEARNING_RATE = 2e-4
NEGATIVE_WEIGHT = 0.2  # caller/summarizer; planner luôn chỉ SFT
RANKING_BETA = 1.0
RANKING_MARGIN = 0.2
SAVE_STEPS = 100
LOGGING_STEPS = 10
OVERWRITE_ADAPTERS = False
SKIP_COMPLETED_ROLES = True  # Reuse matching completed roles; change output/overwrite after a smoke run
RESUME_FROM = {'planner': None, 'caller': None, 'summarizer': None}
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('Source:',SCRIPT_DIR)
print('Outputs:',WORK_DIR)


## Dependencies và GPU
Giữ PyTorch CUDA của Kaggle. Cell chỉ cài stack train, không cài vLLM. Các job chạy trong subprocess mới để tránh module cũ đã import trong kernel. Nếu muốn đánh giá vLLM sau train, dùng kaggle_runner.ipynb trong session inference riêng.


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
                '-r',str(SCRIPT_DIR/'requirements/finetune.txt')], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật GPU trong Kaggle trước khi train Qwen3-4B.')
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
print('BF16 supported:',torch.cuda.is_bf16_supported())
print('Base mode:', '4-bit + LoRA' if USE_4BIT else 'FP16/BF16 + LoRA')


## Tạo dữ liệu theo role
Split theo episode trước khi sinh mọi biến thể. Replay gold tạo observation thật đúng thời điểm; future tool results và final_state không vào input. Mẫu âm gồm sai ID/value, lỗi env có thể sửa, write sai nhưng env chấp nhận, và plan thiếu bước.

Gold tool_calls=[] không cho biết đó là no-op hay unsupported, nên không tự gán done/give_up cho summarizer ở các episode này. Planner vẫn học tools=[]. Episode gold không replay thành công được ghi vào excluded_episodes.json.


In [ ]:
command = [sys.executable,str(SCRIPT_DIR/'prepare_data.py'),
           '--output-dir',str(DATA_DIR),'--seed',str(SEED),
           '--validation-fraction',str(VALIDATION_FRACTION),
           '--limit',str(DATA_LIMIT),'--negatives-per-example',str(NEGATIVES_PER_EXAMPLE),
           '--counterfactuals-per-case',str(COUNTERFACTUALS_PER_CASE)]
if DATA_ROOT:
    command += ['--data-root',str(DATA_ROOT)]
if HOLDOUT_IDS:
    command += ['--holdout-ids',str(HOLDOUT_IDS)]
print('Running:', ' '.join(command))
subprocess.run(command,check=True)


In [ ]:
manifest = json.loads((DATA_DIR/'manifest.json').read_text())
splits = json.loads((DATA_DIR/'split_ids.json').read_text())
assert not (set(splits['train']) & set(splits['validation']))
assert not (set(splits['train']) & set(splits['holdout']))
assert not (set(splits['validation']) & set(splits['holdout']))
print(json.dumps(manifest,ensure_ascii=False,indent=2))
print('Excluded gold episodes:',json.loads((DATA_DIR/'excluded_episodes.json').read_text()))
print('LoRA output dirs:',*[str(ADAPTER_ROOT/r) for r in TRAIN_ROLES],sep='\n')


## Train ba local LoRA
Mỗi role load lại MODEL_ID gốc, tạo LoRA mới; không lấy adapter role trước làm điểm khởi tạo. Frozen base chỉ đọc, train q/k/v/o và gate/up/down projections.

Loss = SFT assistant-only + NEGATIVE_WEIGHT × softplus(margin − beta × (mean_logp_chosen − mean_logp_rejected)). Đây là **pairwise ranking không reference**, không gọi là DPO. Planner không dùng ranking. Prompt/padding không tính loss; summarizer mask answer.

Không cắt target hoặc bỏ nửa JSON để vừa context: ví dụ quá MAX_SEQ_LENGTH bị loại và thống kê trong config. Xem số used/overlength của từng role. Nên chạy MAX_STEPS=5-10 trước; sau đó chọn output mới hoặc bật overwrite/resume có chủ đích.

Nếu có CUDA illegal memory access: restart session khi process/kernel còn giữ CUDA context bị lỗi, bật CUDA_DEBUG=True và chạy lại role bị lỗi với MAX_STEPS=5 trong output mới. Mỗi role lưu toàn bộ stdout/stderr vào training_runs/<role>.console.log và thông tin môi trường vào <role>.runtime.json. Debug không tự sửa kernel, thay loss hoặc tắt dropout. Tắt CUDA_DEBUG sau chẩn đoán.


In [ ]:
import hashlib
for role in TRAIN_ROLES:
    dest=ADAPTER_ROOT/role
    completed=all((dest/name).is_file() for name in
                  ('adapter_config.json','adapter_model.safetensors','role_metadata.json','training_metrics.json'))
    if completed and SKIP_COMPLETED_ROLES and not OVERWRITE_ADAPTERS and not RESUME_FROM.get(role):
        meta=json.loads((dest/'role_metadata.json').read_text())
        prompt_name={'planner':'planner.txt','caller':'caller.txt','summarizer':'summarizer.txt'}[role]
        digest=hashlib.sha256((SCRIPT_DIR/'src/lifemate_v8_finetune/prompts'/prompt_name).read_bytes()).hexdigest()
        manifest_digest=hashlib.sha256(json.dumps(manifest,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
        assert meta['role']==role and meta['base_model']==MODEL_ID and meta['prompt_sha256']==digest
        assert meta['dataset_manifest_sha256']==manifest_digest, 'Completed adapter uses a different dataset; choose output/overwrite.'
        print('Reusing completed role:',dest,flush=True)
        continue
    command = [sys.executable,str(SCRIPT_DIR/'train_role.py'),
        '--role',role,'--data-dir',str(DATA_DIR),'--output-dir',str(TRAIN_OUTPUT),
        '--model',MODEL_ID,'--seed',str(SEED),'--epochs',str(EPOCHS),
        '--max-steps',str(MAX_STEPS),'--max-seq-length',str(MAX_SEQ_LENGTH),
        '--batch-size',str(BATCH_SIZE),'--gradient-accumulation',str(GRADIENT_ACCUMULATION),
        '--learning-rate',str(LEARNING_RATE),'--lora-r',str(LORA_R),
        '--lora-alpha',str(LORA_ALPHA),'--lora-dropout',str(LORA_DROPOUT),
        '--negative-weight',str(NEGATIVE_WEIGHT),'--ranking-beta',str(RANKING_BETA),
        '--ranking-margin',str(RANKING_MARGIN),'--save-steps',str(SAVE_STEPS),
        '--logging-steps',str(LOGGING_STEPS)]
    if CUDA_DEBUG:
        command.append('--cuda-debug')
    if not USE_4BIT:
        command.append('--full-precision-base')
    if OVERWRITE_ADAPTERS:
        command.append('--overwrite-adapter')
    if RESUME_FROM.get(role):
        command += ['--resume-from-checkpoint',str(RESUME_FROM[role])]
    print('Training role:',role,flush=True)
    print('Command:', ' '.join(command),flush=True)
    run_env = os.environ.copy()
    run_env['PYTHONUNBUFFERED'] = '1'
    if CUDA_DEBUG:
        run_env['CUDA_LAUNCH_BLOCKING'] = '1'
    log_path = TRAIN_OUTPUT/'training_runs'/f'{role}.console.log'
    log_path.parent.mkdir(parents=True,exist_ok=True)
    print('Full console log:',log_path,flush=True)
    with log_path.open('w',encoding='utf-8') as log:
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1,env=run_env) as process:
            for line in process.stdout:
                print(line,end='',flush=True)
                log.write(line)
                log.flush()
            code=process.wait()
    if code:
        print('Training failed. Preserve the full log:',log_path,flush=True)
        raise subprocess.CalledProcessError(code,command)
    print('Saved:',ADAPTER_ROOT/role,flush=True)


## Kiểm tra và đóng gói adapter
Validation loss/preference accuracy chỉ kiểm target trong data; chưa thay thế đánh giá workflow. Tokenizer lưu cùng adapter, không có full Qwen weights trong ZIP này. Checkpoint optimizer để resume ở local_lora/checkpoints, không đưa vào ZIP adapter.


In [ ]:
import zipfile
for role in TRAIN_ROLES:
    p=ADAPTER_ROOT/role
    for name in ('adapter_config.json','adapter_model.safetensors','role_metadata.json'):
        if not (p/name).is_file():
            raise FileNotFoundError(p/name)
    print(role,json.loads((p/'training_metrics.json').read_text()))
if all((ADAPTER_ROOT/r/'role_metadata.json').is_file() for r in ('planner','caller','summarizer')):
    archive=WORK_DIR/'lora_adapters.zip'
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(ADAPTER_ROOT.rglob('*')):
            if p.is_file():
                z.write(p,Path('adapters')/p.relative_to(ADAPTER_ROOT))
    print('Download:',archive)


## Tùy chọn: đánh giá toàn workflow trên đúng holdout
Mặc định chưa chạy để bạn chủ động thời gian. Transformers dùng một base + ba PEFT adapters, workers=1; vLLM inference/batch dùng notebook kaggle_runner.ipynb và ADAPTER_ROOT. Runtime kiểm role, base model và prompt hash trước khi load.

Dùng holdout_ids.json, không lấy random limit khác rồi vô tình chấm lại train. Nếu DATA_LIMIT chỉ smoke build, kết quả adapter chưa có ý nghĩa benchmark.


In [ ]:
RUN_WORKFLOW_EVAL = False
command=[sys.executable,str(SCRIPT_DIR/'run.py'),'--mode','workflow',
    '--llm-backend','transformers','--llm-model',MODEL_ID,
    '--adapter-root',str(ADAPTER_ROOT),'--sample-ids-file',str(DATA_DIR/'holdout_ids.json'),
    '--seed',str(SEED),'--inference-workers','1','--output-dir',str(WORK_DIR/'eval_holdout')]
if DATA_ROOT:
    command += ['--data-root',str(DATA_ROOT)]
print('Evaluation command:', ' '.join(command))
if RUN_WORKFLOW_EVAL:
    if not splits['holdout']:
        raise ValueError('Configure a nonempty holdout before workflow evaluation.')
    subprocess.run(command,check=True)
